In [1]:
# In[1]:


# =============================================================================
# CONFIG + CONSTANTS  (names per Fabric_Naming_Convention_Guidelines.pdf)
# =============================================================================
import threading
from pyspark.sql import functions as F
from delta.tables import DeltaTable

GOLD_SCHEMA = "lh_jde_gold.rpt"

# ── refresh / runtime config (CDF concept adopted from ESO4 / nb_silver_to_gold_eso7_v2) ──
ENV             = "dev"
TRIGGER         = {"processingTime": "30 seconds"}
CKPT            = f"Files/checkpoints/eso5_fact_{ENV}"   # OWN root — independent of other notebooks
OVERWRITE       = True    # ⚠ ONE-OFF full reprocess — set back to False after a healthy run

# TRUE ⇒ the notebook contains NO row-selecting predicate ANYWHERE: leg B takes the WHOLE of F4311, not
# just the `PDDCTO='OX' AND PDLITM='HOLADD'` lines report 4 consumes. Those two conditions become the
# `row_class` calculation instead ('PO_HOLADD' vs 'PO_OTHER'), and `item_number` (PDLITM) + `po_order_type`
# (PDDCTO) are on every PO row, so the report filters to OX/HOLADD itself.
# ⚠ VOLUME: the fact is then F4211 ∪ F4311 in full. The 'PO_OTHER' rows are carried for filtering symmetry;
#   none of the five reports reads them. Set False to restore the OX/HOLADD predicate. See design §3d.
PO_LEG_UNFILTERED = True

_FACT_LOCK      = threading.Lock()   # serialises fact writes across the foreachBatch threads

# ── report scaling ──────────────────────────────────────────────────────────────
# Hubble de-scaled RAW JDE integers: qty /1000 (3 implied dec), rate ABURAT*0.01, and — in the HOLADD
# variation only — SDUPRC/1000000 and SDAEXP/100. Those divisors are all implied-decimal decoding, which
# SILVER HAS ALREADY DONE, so they all drop out here and the two queries' scaling agrees. Only the
# BUSINESS factors survive: qty(COM) = units / 2000 (tons). (design §5 — confirm scaling.)
RATE_FACTOR     = 1.0     # ABURAT already decoded (Hubble *0.01 not needed)
TONS_DIVISOR    = 2000.0  # COM lines: units → tons

# ── Silver sources ──────────────────────────────────────────────────────────────
SRC_SCHEMA    = "jde_cdc"     # Silver Change Data Feed schema; CDF must be ON for F4211 *and* F4311.
SRC_LAKEHOUSE = "lh_jde_silver"
F4211_TBL    = "f4211_sales_order_detail_file"                    # streamed (the load/order-line driver)
F4311_TBL    = "f4311_purchase_order_detail_file"                 # streamed (OX status/amount + PO_HOLADD rows)
F554201T_TBL = "f554201t_sand_box_sales_order_qc_information"     # static (Sand PO Number / QC / legs)
F0911_TBL    = "f0911_account_ledger"                             # static (Carrier PO GL Post flag)
F43121_TBL   = "f43121_purchase_order_receiver_file"              # static (PO Receipt GL Date / GLPost doc)
F0101_TBL    = "f0101_address_book_master"                        # static — LOFA rate (ABURAT) only
F4201_TBL    = "f4201_sales_order_header_file"                    # static — report-5 header attrs (SHMCU/SHAN8/…)
# NOTE: F0005 is NOT read here. Its 55/UP attributes live in the Gold dim below.

# ── Gold READ (prerequisite dim — built by nb_eso5_gold_dim_uss_plant) ───────────
T_DIM_USS_PLANT = f"{GOLD_SCHEMA}.dim_uss_plant"

# ── Gold target BUILT here ──────────────────────────────────────────────────────
T_FACT      = f"{GOLD_SCHEMA}.fact_extended_sales_order_5"

print(f"ESO5 Gold fact processor — trigger {TRIGGER}  target {T_FACT}")

StatementMeta(, d3ebc738-5c6a-420e-b066-84b1d4d5928b, 3, Finished, Available, Finished, False)

ESO5 Gold fact processor — trigger {'processingTime': '30 seconds'}  target lh_jde_gold.rpt.fact_extended_sales_order_5


In [2]:
# In[2]:


# =============================================================================
# HELPERS
# =============================================================================
_SOFT_DELETE_COLS = ["is_delete", "deleted_date_time"]

# Every Silver column name below is taken from eso5/full_metadata.txt (the JDE alias → snake_case map).
# It is NOT literal — read the names off that file rather than transliterating the JDE alias. The ones
# that caught me out: SDITWT=amount_unit_weight, SDSRP1=sales_reporting_code_01, PDUOM/SDUOM=uom_as_input,
# QCLGL1/2/3=descriptn_01/02/03, QCFSTR3=future_use_string_03.
def sname(table_name):
    return f"{SRC_LAKEHOUSE}.{SRC_SCHEMA}.{table_name}"

def load_silver_table(table_name):
    df = spark.table(sname(table_name))
    if "is_delete" in df.columns:
        df = df.filter(F.col("is_delete") == 0)
    return df.select(*[c for c in df.columns if c not in _SOFT_DELETE_COLS])

def sk(*cols):
    return F.sha2(F.concat_ws("||", *[F.col(c).cast("string") if isinstance(c, str) else c.cast("string")
                                       for c in cols]), 256)

def load_scope_expr(kcoo_col, dcto_col, doco_col):
    """The CDC delete scope (the load). Built TWICE — once from the fact's own columns (trimmed strings,
    int64 load_number) and once from the RAW Silver CDF columns (padded strings, double DOCO) — so it
    MUST normalise both sides identically. Without the trim + long cast the two sha2 inputs differ
    ("00750||SX ||1184310.0" vs "00750||SX||1184310"), the scope delete matches nothing, and every
    update APPENDS a duplicate load instead of replacing it."""
    return sk(F.trim(F.col(kcoo_col)), F.trim(F.col(dcto_col)), F.col(doco_col).cast("long"))

def current_version(silver_table):
    return spark.sql(f"DESCRIBE HISTORY {sname(silver_table)}").select(F.max("version")).first()[0]

def load_uss_plant_mcu():
    """Gold dim_uss_plant → (vendor_number, lofa_mcu). The ONLY F0005-derived value the fact needs at
    build time (the SOORDERNO SO-match MCU). Read from the DIM, never from Silver F0005."""
    if not spark.catalog.tableExists(T_DIM_USS_PLANT):
        raise RuntimeError(
            f"{T_DIM_USS_PLANT} not found — run nb_eso5_gold_dim_uss_plant FIRST. "
            "The fact needs its lofa_mcu (F0005 55/UP DRSPHD) for the SBXUSSSAND SOORDERNO match.")
    # vendor_number is DOUBLE (it must match the Double fact FK for the Direct Lake relationship);
    # the join below casts both sides to long so the comparison is exact.
    return (spark.table(T_DIM_USS_PLANT)
            .select(F.col("vendor_number").alias("u_vend"),
                    F.trim(F.col("lofa_mcu")).alias("u_mcu"))
            .where(F.col("u_vend").isNotNull())
            .dropDuplicates(["u_vend"]))

StatementMeta(, d3ebc738-5c6a-420e-b066-84b1d4d5928b, 4, Finished, Available, Finished, False)

In [4]:
# In[3]:


# =============================================================================
# CDC WRITE HELPER — NO audit columns (CDF concept from ESO4)
#   fact : delete the affected LOAD scope, then APPEND freshly recomputed lines (handles
#          insert/update/delete uniformly); writes serialised by _FACT_LOCK across BOTH streams.
# =============================================================================
def _write_new_table(df, target, cdf=True):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_SCHEMA}")
    w = df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
    if cdf:
        w = w.option("delta.enableChangeDataFeed", "true")   # Gold CDF on for downstream
    w.saveAsTable(target)

def recompute_fact(loads):
    """CDC for the fact: delete the affected LOAD scope then append the recomputed lines.
    `loads` = distinct company_key_order_no / order_type / document_order_invoice_e.
    Recomputing a load rebuilds ALL of its row classes (LINE / HOLADD / TEXT / PO_HOLADD) together,
    which is why an F4311-only change can be routed through the very same path."""
    if loads.rdd.isEmpty():
        return 0
    src = transform_fact(restrict_loads=loads)
    with _FACT_LOCK:
        if not spark.catalog.tableExists(T_FACT):
            _write_new_table(src, T_FACT)
            return src.count()
        scope = loads.select(load_scope_expr("company_key_order_no", "order_type",
                                            "document_order_invoice_e")
                             .alias("load_scope_key")).distinct()
        (DeltaTable.forName(spark, T_FACT).alias("t")
            .merge(scope.alias("s"), "t.load_scope_key = s.load_scope_key")
            .whenMatchedDelete().execute())               # drop the load's old lines (all classes)
        src.write.format("delta").mode("append").saveAsTable(T_FACT)   # append current lines
    return src.count()

StatementMeta(, d3ebc738-5c6a-420e-b066-84b1d4d5928b, 6, Finished, Available, Finished, False)

In [5]:
# In[4]:


# =============================================================================
# FACT  fact_extended_sales_order_5  — ONE table, LINE grain, serving all five reports.
#   Grain = one F4211 SX order line (any item, any line type) PLUS one row per orphan F4311 OX
#   HOLADD PO line. `row_class` says which report(s) a row belongs to (see the header block).
# =============================================================================
# Display columns STORED on the fact — the GROUP BY grain. FOUR docx columns are deliberately NOT
# stored (star schema — resolved via dimensions instead):
#   loading_facility_name  → dim_address_loading_facility.name_alpha (reused dim_address_book, F0101)
#   uss_plant_sand / shipped_from / lofa_mcu → dim_uss_plant.* (from F0005 55/UP, keyed by vendor)
FACT_CORE_COLS = [
    "load_number",          # SDDOCO
    "document_type",        # SDDCTO
    "company",              # SDKCOO
    "district",             # SDMCU
    "sold_to",              # SDAN8  (FK → dim_address_sold_to)
    "ship_to",              # SDSHAN (FK → dim_address_ship_to)
    "carrier",              # SDCARS (FK → dim_address_carrier)
    "customer_po",          # SDVR01
    "sand_po_number",       # F554201T QCDS50
    "uss_customer_po",      # SBXUSSSAND SOPONO
    "item_number",          # SDLITM
    "item_description",     # SDDSC1
    "order_date",           # SDTRDJ
    "gl_date",              # SDDGL
    "loading_facility",     # LOFA=SDVEND (FK → dim_address_loading_facility + dim_uss_plant)
    "uss_match",            # SBXUSSSAND MATCHFLAG
    "uss_so_order_no",      # SBXUSSSAND SOORDERNO
    "uss_so_weight",        # SBXUSSSAND SOWEIGHT
    "sbx_weight",           # SBXUSSSAND SXWEIGHT
    "so_alt_bol_no",        # SBXUSSSAND SOALTBOLNO
    "sand_ticket",          # SBXLOADPOVIEW SANDTKT
    "bol",                  # SBXLOADPOVIEW BOL
    "uom",                  # SDUOM
    "quantity",             # QTY (derived)
    "unit_price",           # SDUPRC
    "total_amount",         # SDAEXP
    "last_status",          # SDLTTR
    "next_status",          # SDNXTR
    "invoice_number",       # SDDOC
    "ox_last_status",       # F4311 OXLTTR
    "ox_next_status",       # F4311 OXNXTR
    "ox_amount",            # F4311 OXAMT
    "carrier_po_gl_post_flag",  # F0911 GLPOST
    "po_receipt_gl_date",   # F43121 GLDGJ
    "line_id",              # SDLNID
]
# STATUS — every status field the five queries touch, stored so the REPORT can do all status filtering.
# NO status predicate filters a row anywhere in this notebook (design §7d). The three below are the ones
# that used to BE filters and are now fields; last_status / next_status / ox_last_status / ox_next_status /
# load_last_status are already in the lists above.
FACT_STATUS_COLS = [
    "next_status_num",       # numeric copy of next_status for the core report's SDNXTR<'581' page filter.
                             # Hubble compares SDNXTR as a STRING (only safe for uniform 3-char codes); this
                             # integer column makes `next_status_num < 581` unambiguous in Direct Lake
                             # (calculated columns are forbidden, so the numeric form must be physical).
    "po_holadd_superseded",  # 'Y' ⇔ the load has a LIVE (last_status<>'980') SX HOLADD sales line.
                             # WAS report 4's `NOT EXISTS(...)`. Hubble's orphan set = 'N'.
    "po_order_type",         # PDDCTO on a PO row (document_type is forced to 'SX' by the query); NULL on
                             # F4211 rows. Lets the report filter the PO leg by its real document type.
    "load_max_last_status",  # MXLTTR — recon view's per-load MAX(SDLTTR)  (input to load_last_status)
    "load_min_last_status",  # MILTTR — recon view's per-load MIN(SDLTTR)  (input to load_last_status)
    "ox_amount_gross",       # the F4311 OX money with NO item/status condition — lets the report override
                             # ox_amount's baked-in `item='FRT' | (item='HOLADD' AND last_status<>'980')`
]
# Added so the FOUR Filter-Capture variations can be served from these same rows.
FACT_VARIATION_COLS = [
    "row_class",            # LINE | HOLADD | TEXT | PO_HOLADD  — the per-report row filter
    # report 5 (Reconciliation) pivot INPUTS — its SANDWEIGHT/EXTWEIGHT/MILES/LOFADET/WELLDET/...PP/
    # ...PB/FRTAMT/FSCAMT/SANDAMT/HOLAMT are DAX SUMs of these, filtered by item/category.
    "line_type",            # SDLNTY  ('TL' = the text lines the core view drops)
    "product_category",     # SDPRP1  ('COM' sand, 'FRT' freight)
    "sales_report_code_01", # SDSRP1  ('352' → SANDAMT)
    "units_ordered",        # SDUORG  — RAW decoded units (quantity above is the COM→tons version)
    "item_weight",          # SDITWT  → EXTWEIGHT
    "load_last_status",     # report 5's per-load SDLTTR CASE (see _load_last_status)
    # report 5 F554201T columns beyond QCDS50
    "leg_1",                # QCLGL1
    "leg_2",                # QCLGL2
    "leg_3",                # QCLGL3
    "qc_string_3",          # QCFSTR3
    # report 5 groups by the F4201 HEADER attributes, which need not equal the line's own values
    "header_district",      # SHMCU
    "header_sold_to",       # SHAN8
    "header_ship_to",       # SHSHAN
    "header_carrier",       # SHCARS
    "header_customer_po",   # SHVR01
    "header_order_date",    # SHTRDJ
]
FACT_GROUP_BY_COLS = FACT_CORE_COLS + FACT_STATUS_COLS + FACT_VARIATION_COLS
# SUMmed measure (Hubble ReportColumn1). Semi-additive (constant per LOFA) — see design §5.
FACT_MEASURE_COLS = ["lofa_rate"]
FACT_BUSINESS_COLS = FACT_GROUP_BY_COLS + FACT_MEASURE_COLS


def _pad30(c):
    """rpad(rtrim(rtrim(x,' '),'.'), 30, ' ') — trim trailing spaces then trailing dots, pad to 30."""
    return F.rpad(F.regexp_replace(F.rtrim(c), r"\.+$", ""), 30, " ")

def _pad25(c):
    """rpad(rtrim(x), 25, ' ')."""
    return F.rpad(F.rtrim(c), 25, " ")


# ── the two row legs ────────────────────────────────────────────────────────────
# Both legs are projected into ONE common intermediate schema so the whole downstream join chain
# (BOL / SANDTKT / OX / F43121 / F0911 / F554201T / SBXUSSSAND / SXWEIGHT / F0101 / F4201) is written
# once and applied to both. Leg-B's PO-side OX values ride along as _pox_* and win in the final select.
_LEG_COLS = ["l_kcoo", "l_doco", "l_dcto", "l_mcu", "l_an8", "l_shan", "l_cars", "l_vr01",
             "l_litm", "l_dsc1", "l_uom", "l_uorg", "l_itwt", "l_prp1", "l_srp1", "l_lnty",
             "l_trdj", "l_vend", "l_uprc", "l_aexp", "l_dgl", "l_lttr", "l_nxtr", "l_doc", "l_lnid",
             "row_class", "po_holadd_superseded", "l_po_dcto", "_pox_lttr", "_pox_nxtr", "_pox_amt"]

def _f4211_lines(f4211):
    """Leg A — the F4211 sales-order lines. NO FILTER OF ANY KIND (user direction 2026-07-20). This leg is
    the WHOLE of Silver F4211; every predicate the five queries put in a WHERE is carried as a COLUMN and
    filtered in the Power BI report instead:
        SDDCTO='SX'      -> the `document_type` column   (report filter)
        SDKCOO='00750'   -> the `company` column         (report filter)
        SDLNTY<>'TL'     -> row_class 'TEXT'             (report filter)
        SDLITM<>'HOLADD' -> row_class 'HOLADD'           (report filter)
    The last two could never have been applied anyway: they CONFLICT across the five reports (1-3 drop TL
    and HOLADD, 4 REQUIRES HOLADD, 5 keeps both). That conflict is the structural reason one fact can serve
    five reports only if the discrimination happens at report level — which is exactly what `row_class` is.
    ⚠ With this leg unfiltered, the SO-match / `_load_aggregates` / OX helpers now read the SAME population
    they always needed: the SO leg lives on the SDDCTO='SO' AND SDCO='00400' rows that an SX filter removes.
    See design §7b (the filter contract)."""
    sd = f4211
    row_class = (F.when(F.trim(F.col("line_type")) == "TL", F.lit("TEXT"))
                  .when(F.trim(F.col("identifier_second_item")) == "HOLADD", F.lit("HOLADD"))
                  .otherwise(F.lit("LINE")))
    return sd.select(
        F.trim(F.col("company_key_order_no")).alias("l_kcoo"),
        F.col("document_order_invoice_e").alias("l_doco"),
        F.trim(F.col("order_type")).alias("l_dcto"),
        F.trim(F.col("cost_center")).alias("l_mcu"),
        F.col("address_number").alias("l_an8"),
        F.col("address_number_ship_to").alias("l_shan"),
        F.col("carrier").alias("l_cars"),
        F.col("reference_01").alias("l_vr01"),
        F.trim(F.col("identifier_second_item")).alias("l_litm"),
        F.col("description_line_01").alias("l_dsc1"),
        F.col("uom_as_input").alias("l_uom"),
        F.col("units_transaction_qty").alias("l_uorg"),                     # SDUORG
        F.col("amount_unit_weight").cast("double").alias("l_itwt"),         # SDITWT
        F.trim(F.col("purchasing_report_code_01")).alias("l_prp1"),         # SDPRP1
        F.trim(F.col("sales_reporting_code_01")).alias("l_srp1"),           # SDSRP1
        F.trim(F.col("line_type")).alias("l_lnty"),                         # SDLNTY
        F.col("date_transaction_julian").alias("l_trdj"),
        F.col("primary_last_vendor_no").alias("l_vend"),
        F.col("amt_price_per_unit_02").alias("l_uprc"),
        F.col("amount_extended_price").alias("l_aexp"),
        F.col("dt_for_gl_and_vouch_01").alias("l_dgl"),
        F.trim(F.col("status_code_last")).alias("l_lttr"),
        F.trim(F.col("status_code_next")).alias("l_nxtr"),
        F.col("doc_voucher_invoice_e").alias("l_doc"),
        F.col("line_number").alias("l_lnid"),
        row_class.alias("row_class"),
        F.lit("N").alias("po_holadd_superseded"),   # only ever 'Y' on a PO_HOLADD row (leg B)
        F.lit(None).cast("string").alias("l_po_dcto"),   # PDDCTO — PO rows only
        F.lit(None).cast("string").alias("_pox_lttr"),
        F.lit(None).cast("string").alias("_pox_nxtr"),
        F.lit(None).cast("double").alias("_pox_amt"))

def _po_lines(f4311, la):
    """Leg B — the F4311 purchase-order lines. Report 4's second UNION branch is the OX HOLADD ones; with
    PO_LEG_UNFILTERED (the default) this leg takes the WHOLE table and `row_class` classifies each row
    instead, so the notebook holds NO row-selecting predicate at all:
        row_class = 'PO_HOLADD'  ⇔  PDDCTO='OX' AND PDLITM='HOLADD'   → report 4
                    'PO_OTHER'   ⇔  every other purchase-order line   → no report reads it
    Sales-side attributes are back-filled from the load's FRT sales line, exactly as the query's correlated
    MAX(...) subqueries do, and UPRC/EXTAMT are forced to 0 — the money is on OXAMT. The back-fill and the
    ex-NOT-EXISTS flag both come from `la` (_load_aggregates), so this leg reads F4211 not at all; a PO line
    with no SX load behind it simply gets NULLs.

    ⚠ Hubble's `NOT EXISTS(live SX HOLADD, SDLTTR<>'980')` is GONE — a status test that decided whether a
    row EXISTS. Every HOLADD PO line is a row now, and the test is a FIELD:
        po_holadd_superseded = 'Y'  ⇔  the load already carries a live SX HOLADD sales line
    Hubble's orphan set is exactly `po_holadd_superseded = 'N'`, which report 4 filters on.

    ⚠ `document_type` is forced to 'SX' on every row here — the query hard-codes `'SX' DCTO` so the UNION
    lines up with the sales load, and `load_scope_key` depends on it (a PO change must land in the SX
    load's CDC scope). The PO's OWN document type is kept in `po_order_type` (PDDCTO)."""
    po = f4311 if PO_LEG_UNFILTERED else f4311.where(
        (F.trim(F.col("order_type")) == "OX") & (F.trim(F.col("identifier_2nd_item")) == "HOLADD"))

    # Back-fill + the ex-NOT-EXISTS flag come from the load's SX aggregates (Hubble correlates the FRT
    # subqueries on PDKCOO/PDDOCO with SDDCTO='SX' — hence the literal 'SX' in the join key).
    j = (po.alias("pd")
         .join(la, (F.trim(F.col("pd.company_key_order_no")) == F.col("_la_kcoo")) &
                   (F.col("pd.document_order_invoice_e") == F.col("_la_doco")) &
                   (F.col("_la_dcto") == F.lit("SX")), "left"))
    return j.select(
        F.trim(F.col("pd.company_key_order_no")).alias("l_kcoo"),
        F.col("pd.document_order_invoice_e").alias("l_doco"),
        F.lit("SX").alias("l_dcto"),                                   # the query hard-codes 'SX'
        F.trim(F.col("pd.cost_center")).alias("l_mcu"),                # PDMCU
        F.col("_fr_an8").alias("l_an8"),                               # from the FRT line
        F.col("pd.address_number_ship_to").cast("double").alias("l_shan"),   # PDSHAN
        F.col("pd.address_number").alias("l_cars"),                    # PDAN8 IS the carrier
        F.col("_fr_vr01").alias("l_vr01"),
        F.trim(F.col("pd.identifier_2nd_item")).alias("l_litm"),
        F.col("pd.description_line_01").alias("l_dsc1"),               # PDDSC1
        F.col("pd.uom_as_input").alias("l_uom"),                       # PDUOM
        F.col("pd.units_transaction_qty").cast("double").alias("l_uorg"),    # PDUORG
        F.lit(None).cast("double").alias("l_itwt"),
        F.lit(None).cast("string").alias("l_prp1"),
        F.lit(None).cast("string").alias("l_srp1"),
        F.lit(None).cast("string").alias("l_lnty"),
        F.col("pd.date_transaction_julian").cast("date").alias("l_trdj"),    # PDTRDJ
        F.col("_fr_vend").alias("l_vend"),                             # LOFA from the FRT line
        F.lit(0.0).alias("l_uprc"),                                    # UPRC = 0
        F.lit(0.0).alias("l_aexp"),                                    # EXTAMT = 0
        F.col("_fr_dgl").alias("l_dgl"),
        F.col("_fr_lttr").alias("l_lttr"),
        F.col("_fr_nxtr").alias("l_nxtr"),
        F.col("_fr_doc").alias("l_doc"),
        F.col("pd.line_number").cast("double").alias("l_lnid"),        # PDLNID
        # `PDDCTO='OX' AND PDLITM='HOLADD'` — Hubble's WHERE for this UNION branch. With
        # PO_LEG_UNFILTERED it is no longer a filter: it CLASSIFIES the row. Only 'PO_HOLADD' rows feed
        # report 4; 'PO_OTHER' is every other purchase-order line, carried but read by no report.
        F.when((F.trim(F.col("pd.order_type")) == "OX") &
               (F.trim(F.col("pd.identifier_2nd_item")) == "HOLADD"), F.lit("PO_HOLADD"))
         .otherwise(F.lit("PO_OTHER")).alias("row_class"),
        F.coalesce(F.col("_live_holadd"), F.lit("N")).alias("po_holadd_superseded"),   # ex-NOT EXISTS
        F.trim(F.col("pd.order_type")).alias("l_po_dcto"),             # PDDCTO (document_type is forced
                                                                       # to 'SX' by the query, so the PO's
                                                                       # own type needs its own column)
        F.trim(F.col("pd.status_code_last")).alias("_pox_lttr"),       # OXLTTR = PDLTTR (row's own)
        F.trim(F.col("pd.status_code_next")).alias("_pox_nxtr"),       # OXNXTR = PDNXTR
        F.col("pd.amount_extended_price").cast("double").alias("_pox_amt"))   # OXAMT = PDAEXP


def _load_aggregates(f4211):
    """EVERY per-load value the five queries compute with a correlated subquery, in ONE pass over F4211.

    ⚠ THE POINT OF THIS FUNCTION: not one of the source predicates (SDLITM='BOL' / 'SANDTKTNBR' / 'HOLADD'
    / 'FRT', SDPRP1='COM', SDLTTR<>'980', SDDCTO='SX') appears in a WHERE. Every one is a CASE inside an
    aggregate, so it decides whether a line CONTRIBUTES TO A VALUE — never whether a row SURVIVES. No row
    is filtered out of anything. That is the difference between a calculation (kept: it IS the business
    logic) and a filter (gone: it belongs to the report).

    Grain (kcoo, doco, dcto) — the group key carries the order type and company, so an SX/00750 row reads
    exactly the values Hubble's `SDDCTO='SX' AND SDKCOO='00750'` subqueries would have produced, with no
    constant hard-coded anywhere."""
    item = F.trim(F.col("identifier_second_item"))
    lttr = F.trim(F.col("status_code_last"))
    return (f4211.groupBy(F.trim(F.col("company_key_order_no")).alias("_la_kcoo"),
                          F.col("document_order_invoice_e").alias("_la_doco"),
                          F.trim(F.col("order_type")).alias("_la_dcto"))
            .agg(
                # SBXLOADPOVIEW: BOL / SANDTKT  (were `WHERE SDLITM = 'BOL' / 'SANDTKTNBR'`)
                F.max(F.when(item == "BOL", F.col("description_line_01"))).alias("bol"),
                F.max(F.when(item == "SANDTKTNBR", F.col("description_line_01"))).alias("sand_ticket"),
                # SBXUSSSAND's M row = the load's SANDTKTNBR line (its vendor drives the 55/UP plant match)
                F.max(F.when(item == "SANDTKTNBR", F.col("primary_last_vendor_no"))).alias("_st_vend"),
                # SXWEIGHT  (was `WHERE SDDCTO='SX' AND SDPRP1='COM' AND SDLTTR<>'980'`)
                F.sum(F.when((F.trim(F.col("purchasing_report_code_01")) == "COM") & (lttr != "980"),
                             F.col("units_transaction_qty"))).alias("sbx_weight"),
                # report 4's `NOT EXISTS(live SX HOLADD, SDLTTR<>'980')` → a FIELD, not a row filter
                F.max(F.when((item == "HOLADD") & (lttr != "980"), F.lit("Y"))).alias("_live_holadd"),
                # report 4 leg B back-fills its sales-side attributes from the load's FRT line
                F.max(F.when(item == "FRT", F.col("reference_01"))).alias("_fr_vr01"),
                F.max(F.when(item == "FRT", F.col("address_number"))).alias("_fr_an8"),
                F.max(F.when(item == "FRT", F.col("primary_last_vendor_no"))).alias("_fr_vend"),
                F.max(F.when(item == "FRT", F.col("dt_for_gl_and_vouch_01"))).alias("_fr_dgl"),
                F.max(F.when(item == "FRT", lttr)).alias("_fr_lttr"),
                F.max(F.when(item == "FRT", F.trim(F.col("status_code_next")))).alias("_fr_nxtr"),
                F.max(F.when(item == "FRT", F.col("doc_voucher_invoice_e"))).alias("_fr_doc"),
                # report 5's per-load SDLTTR CASE + its two inputs (all three stored on the fact)
                F.max(lttr).alias("load_max_last_status"),                                  # MXLTTR
                F.min(lttr).alias("load_min_last_status"),                                  # MILTTR
                F.max(F.when((item != "HOLADD") & (F.col("amount_extended_price") != 0),
                             lttr)).alias("_alt"))
            .withColumn("load_last_status",
                        F.when((F.col("load_max_last_status") == "980") &
                               (F.col("load_min_last_status") == "980"), F.col("load_max_last_status"))
                         .otherwise(F.col("_alt")))
            .drop("_alt"))


def transform_fact(restrict_loads=None):
    f4211  = load_silver_table(F4211_TBL)
    qc     = load_silver_table(F554201T_TBL)
    f4311  = load_silver_table(F4311_TBL)
    f0911  = load_silver_table(F0911_TBL)
    f43121 = load_silver_table(F43121_TBL)
    ab     = load_silver_table(F0101_TBL)
    f4201  = load_silver_table(F4201_TBL)         # report-5 header

    # ── normalize the order/load key type ACROSS tables (bug fix 2026-07-21) ─────────────────────
    # SDDOCO / PDDOCO / PRDOCO are the SAME JDE data item (DOCO), but Silver can land them with
    # DIFFERENT physical types per table (e.g. F4311 as string, F4211/F43121 as decimal). A cross-table
    # equality JOIN on document_order_invoice_e then SILENTLY MISSES — invisible for leg A (all F4211),
    # but on the F4311 PO_HOLADD rows it wipes EVERY sales-side back-fill: sold_to / customer_po /
    # loading_facility / last_status / next_status / invoice_number / bol / sand_ticket / po_receipt_gl_date
    # / carrier_po_gl_post_flag — and gl_date collapses to coalesce(NULL, 1900-01-01). (ox_* survive because
    # they read F4311↔F4311.) Cast the key to ONE canonical type wherever it is carried so all doco joins
    # are long==long. DOCO is an integer order number (0 implied decimals), so the cast is lossless.
    _DOCO = "document_order_invoice_e"
    f4211  = f4211.withColumn(_DOCO,  F.col(_DOCO).cast("long"))
    f4311  = f4311.withColumn(_DOCO,  F.col(_DOCO).cast("long"))
    f43121 = f43121.withColumn(_DOCO, F.col(_DOCO).cast("long"))
    qc     = qc.withColumn(_DOCO,     F.col(_DOCO).cast("long"))
    f4201  = f4201.withColumn(_DOCO,  F.col(_DOCO).cast("long"))

    # ── ALL the per-load F4211 subqueries, in one filter-free pass (bol / sand_ticket / sbx_weight /
    #    load_last_status + inputs / the leg-B FRT back-fill / the ex-NOT-EXISTS flag) ──
    la = _load_aggregates(f4211)

    # ── the row population: F4211 SX/00750 lines  UNION  the WHOLE F4311 PO leg (§7b) ──
    base = (_f4211_lines(f4211).select(*_LEG_COLS)
            .unionByName(_po_lines(f4311, la).select(*_LEG_COLS)))

    # CDC scope restriction — keep only the changed loads' rows (ALL classes of those loads)
    if restrict_loads is not None:
        base = base.join(restrict_loads.alias("rl"),
                         (F.col("l_kcoo") == F.trim(F.col("rl.company_key_order_no"))) &
                         (F.col("l_dcto") == F.trim(F.col("rl.order_type"))) &
                         (F.col("l_doco") == F.col("rl.document_order_invoice_e").cast("long")), "left_semi")

    # ── F554201T — QCDS50 (Sand PO Number) + report 5's QCLGL1/2/3 + QCFSTR3, per (kcoo, doco, dcto).
    #    The legs are Silver `descriptn_01/02/03` and QCFSTR3 is `future_use_string_03` — the JDE→snake_case
    #    map is NOT literal, so read the names off full_metadata.txt rather than guessing from the alias. ──
    def _qcf(src, alias):
        return F.first(F.col(src), ignorenulls=True).alias(alias)
    qcv = (qc.groupBy(F.trim(F.col("company_key_order_no")).alias("qc_kcoo"),
                      F.col("document_order_invoice_e").alias("qc_doco"),
                      F.trim(F.col("order_type")).alias("qc_dcto"))
           .agg(F.first(F.trim(F.col("description_50_characters")), ignorenulls=True).alias("sand_po_number"),
                _qcf("descriptn_01", "leg_1"),                 # QCLGL1
                _qcf("descriptn_02", "leg_2"),                 # QCLGL2
                _qcf("descriptn_03", "leg_3"),                 # QCLGL3
                _qcf("future_use_string_03", "qc_string_3")))  # QCFSTR3

    # ── OX status + OX amount from F4311, by (item, load) — Hubble's correlation.
    #    `PDDCTO='OX'` and `PDKCOO='00750'` were WHERE constants; they are CASE conditions now, so no
    #    F4311 row is filtered away — they merely decide which rows contribute to the OX values. ──
    _is_ox = ((F.trim(F.col("order_type")) == "OX") &
              (F.trim(F.col("company_key_order_no")) == "00750"))
    ox = (f4311.groupBy(F.trim(F.col("identifier_2nd_item")).alias("_ox_item"),
                        F.col("document_order_invoice_e").alias("_ox_doco"))
          .agg(F.max(F.when(_is_ox, F.col("status_code_next"))).alias("_ox_nxtr"),
               F.max(F.when(_is_ox, F.col("status_code_last"))).alias("_ox_lttr"),
               F.sum(F.when(_is_ox, F.col("amount_extended_price"))).alias("_ox_amt")))

    # ── PO Receipt GL Date — F43121, by the line keys. `PRDCT='OV' AND PRMATC='1' AND PRDGL>1` are CASE
    #    conditions, not a WHERE: the value is the MAX over the matching receipts, no row is dropped. ──
    _rc_ok = ((F.trim(F.col("document_type")) == "OV") &
              (F.trim(F.col("match_type")) == "1") &
              (F.col("dt_for_gl_and_vouch_01").isNotNull()))
    recv = (f43121.groupBy(F.col("address_number").alias("_rc_pran8"),
                           F.trim(F.col("company_key_order_no")).alias("_rc_kcoo"),
                           F.col("document_order_invoice_e").alias("_rc_doco"),
                           F.trim(F.col("identifier_2nd_item")).alias("_rc_item"))
            .agg(F.max(F.when(_rc_ok, F.col("dt_for_gl_and_vouch_01"))).alias("po_receipt_gl_date")))

    # ── Carrier PO GL Post flag — F0911 doc ∈ F43121 PRDOC, linked by the line keys. The `GLDCT='OV'`,
    #    `GLKCO='00750'` and `PRDCT='OV'` constants ride inside the CASE; the doc linkage is a JOIN. ──
    recv_docs = f43121.select(F.col("doc_voucher_invoice_e").alias("_gd_prdoc"),
                              F.col("address_number").alias("_gd_pran8"),
                              F.trim(F.col("company_key_order_no")).alias("_gd_kcoo"),
                              F.col("document_order_invoice_e").alias("_gd_doco"),
                              F.trim(F.col("identifier_2nd_item")).alias("_gd_item"),
                              F.trim(F.col("document_type")).alias("_gd_dct"))
    glpost_docs = f0911.select(F.col("doc_voucher_invoice_e").alias("_gl_doc"),
                               F.col("gl_posted_code").alias("_gl_post"),
                               F.trim(F.col("document_type")).alias("_gl_dct"),
                               F.trim(F.col("company_key")).alias("_gl_kco"))
    glpost = (recv_docs.join(glpost_docs, F.col("_gd_prdoc") == F.col("_gl_doc"), "inner")
              .groupBy("_gd_pran8", "_gd_kcoo", "_gd_doco", "_gd_item")
              .agg(F.max(F.when((F.col("_gd_dct") == "OV") & (F.col("_gl_dct") == "OV") &
                                (F.col("_gl_kco") == "00750"), F.col("_gl_post")))
                    .alias("carrier_po_gl_post_flag")))

    # ── SBXUSSSAND — the load's SANDTKTNBR line (from `la`) INNER F554201T on (kcoo, doco, dcto).
    #    Hubble's `M.SDDCTO='SX' AND M.SDLITM='SANDTKTNBR' AND M.SDKCOO='00750'` needs no WHERE here: the
    #    item condition is already the CASE inside `la`, and the order type + company are the join keys,
    #    so an SX/00750 row reads exactly what Hubble's M would have given it. ──
    plant = load_uss_plant_mcu()   # F0005 55/UP DRSPHD, from the GOLD dim — never Silver F0005
    m2f = (la.alias("la")
           .join(qcv.alias("qc"),
                 (F.col("la._la_kcoo") == F.col("qc.qc_kcoo")) &
                 (F.col("la._la_doco") == F.col("qc.qc_doco")) &
                 (F.col("la._la_dcto") == F.col("qc.qc_dcto")), "inner")
           .join(plant, F.col("la._st_vend").cast("long") == F.col("u_vend").cast("long"), "left")
           .select(F.col("la._la_kcoo").alias("us_kcoo"),
                   F.col("la._la_doco").alias("us_doco"),
                   F.col("la._la_dcto").alias("us_dcto"),
                   F.col("la.sand_ticket").alias("us_sddsc1"),      # M.SDDSC1 (the sand-ticket number)
                   F.col("qc.sand_po_number").alias("us_qcds50"),   # F554201T.QCDS50
                   F.col("u_mcu").alias("lofa_mcu")))               # SO-match input only (not stored)

    # SO orders — matched to the sand load by padded pull_signal + reference_01. `L.SDDCTO='SO' AND
    # L.SDCO='00400'` are part of the MATCH, so they live in the JOIN CONDITION, not in a WHERE on F4211.
    so = f4211.select(F.col("pull_signal").alias("so_psig"),
                      F.col("reference_01").alias("so_vr01"),
                      F.col("document_order_invoice_e").cast("long").alias("so_doco"),  # → int64
                      F.col("cost_center").alias("so_mcu"),
                      F.trim(F.col("line_type")).alias("so_lnty"),
                      F.col("units_secondary_qty_or").alias("so_sqor"),
                      F.trim(F.col("order_type")).alias("so_dcto"),
                      F.trim(F.col("company")).alias("so_co"))
    matched = (m2f.alias("u").join(so.alias("s"),
                   (F.col("s.so_psig") == _pad30(F.col("u.us_sddsc1"))) &
                   (F.col("s.so_vr01") == _pad25(F.col("u.us_qcds50"))) &
                   (F.col("s.so_dcto") == "SO") & (F.col("s.so_co") == "00400"), "left"))
    sbxusssand = (matched.groupBy("us_kcoo", "us_doco", "us_dcto")
                  .agg(F.max(F.when(F.col("s.so_doco").isNotNull(), F.lit("Y"))).alias("uss_match"),
                       F.first(F.col("s.so_vr01"),  ignorenulls=True).alias("uss_customer_po"),   # SOPONO
                       F.first(F.col("s.so_psig"),  ignorenulls=True).alias("so_alt_bol_no"),     # SOALTBOLNO
                       # SOORDERNO — SO doco whose district (so_mcu) matches the vendor's plant MCU (DRSPHD)
                       F.first(F.when(F.trim(F.col("s.so_mcu")) == F.col("lofa_mcu"), F.col("s.so_doco")),
                               ignorenulls=True).alias("uss_so_order_no"),
                       # SOWEIGHT — sum secondary qty over matched SO 'S' lines (decoded; no /1000)
                       F.sum(F.when(F.col("s.so_lnty") == "S", F.col("s.so_sqor"))).alias("uss_so_weight")))

    # F0101 loading-facility lookup (LOFA = ABAN8): RATE only. The NAME (ABALPH) resolves via the
    # reused dim_address_loading_facility relationship; F0101 is read here ONLY for ABURAT (the rate),
    # which the reused dim_address_book does not carry (`user_reserved_amount` absent).
    #
    # Hubble's rate source is `SELECT … FROM F0101 WHERE ABAT1 BETWEEN 'A '..'P ' OR 'R '..'ZZZ'` — a
    # search-type band that says WHICH address-book rows COUNT AS a rate source (it excludes the 'Q'
    # band). Following the §3a-bis rule, that predicate is NOT deleted and is NOT a WHERE: it is a CASE
    # INSIDE THE AGGREGATE. Deleting it would not make the rate "unfiltered", it would make it WRONG —
    # a 'Q'-band facility would start reporting a rate Hubble shows as blank. As a CASE it drops nothing:
    # every address_number keeps its group, and an out-of-band facility simply yields a NULL rate,
    # which is exactly what Hubble's LEFT JOIN produced.
    # (`address_type_01` is on the reused `dim_address_loading_facility` if the report ever wants to see
    #  or override the band.)  ABAT1 = address_type_01, rpad to 3 to mirror the padded 'A  '/'P  '/'ZZZ'.
    _at = F.rpad(F.rtrim(F.col("address_type_01")), 3, " ")
    _is_rate_source = (((_at >= F.lit("A  ")) & (_at <= F.lit("P  "))) |
                       ((_at >= F.lit("R  ")) & (_at <= F.lit("ZZZ"))))
    lofa = (ab.groupBy(F.col("address_number").alias("ab_an8"))
            .agg(F.first(F.when(_is_rate_source, F.col("user_reserved_amount")),
                         ignorenulls=True).alias("_aburat")))

    # F4201 sales-order HEADER — report 5 groups by SHMCU/SHAN8/SHSHAN/SHCARS/SHVR01/SHTRDJ, which are
    # header values and need not equal the line's own.
    hdr = (f4201.select(
                F.trim(F.col("company_key_order_no")).alias("h_kcoo"),          # SHKCOO
                F.col("document_order_invoice_e").alias("h_doco"),              # SHDOCO
                F.trim(F.col("order_type")).alias("h_dcto"),                    # SHDCTO
                F.trim(F.col("cost_center")).alias("header_district"),          # SHMCU
                F.col("address_number").cast("double").alias("header_sold_to"),         # SHAN8
                F.col("address_number_ship_to").cast("double").alias("header_ship_to"), # SHSHAN
                F.col("carrier").cast("double").alias("header_carrier"),        # SHCARS
                F.col("reference_01").alias("header_customer_po"),              # SHVR01
                F.col("date_transaction_julian").cast("date").alias("header_order_date"))  # SHTRDJ
           .dropDuplicates(["h_kcoo", "h_doco", "h_dcto"]))

    # ── derivations on the common line ──────────────────────────────────────────────
    # QTY = DECODE(SDPRP1,'COM',(SDUORG/1000)/2000, SDUORG/1000) — Silver is decoded, so only the
    # COM→tons factor survives. PO_HOLADD rows have no SDPRP1, so they take the plain-units branch,
    # which is what report 4's `pduorg / 1000` reduces to.
    qty = F.when(F.col("l_prp1") == "COM", F.col("l_uorg") / F.lit(TONS_DIVISOR)).otherwise(F.col("l_uorg"))
    gl_date = F.coalesce(F.col("l_dgl"), F.to_date(F.lit("1900-01-01")))
    # OXAMT — the core query charges it to the FRT line; the HOLADD variation charges it to a live
    # HOLADD line (SDLTTR<>'980'); a PO_HOLADD row carries its own PDAEXP. Union of the three rules.
    # This is a CASE (a calculation), not a filter — no row is dropped. But it does BAKE IN a status rule,
    # so `ox_amount_gross` below exposes the same money with NO item/status condition, letting the report
    # own the status decision entirely (`last_status` is on the row).
    _is_po_row = F.col("row_class").isin("PO_HOLADD", "PO_OTHER")
    ox_amount = (F.when(_is_po_row, F.coalesce(F.col("_pox_amt"), F.lit(0.0)))
                  .when(F.col("l_litm") == "FRT", F.coalesce(F.col("_ox_amt"), F.lit(0.0)))
                  .when((F.col("l_litm") == "HOLADD") & (F.col("l_lttr") != "980"),
                        F.coalesce(F.col("_ox_amt"), F.lit(0.0)))
                  .otherwise(F.lit(0.0)))
    ox_amount_gross = F.coalesce(F.col("_pox_amt"), F.col("_ox_amt"), F.lit(0.0))

    # ── assemble: base LEFT (per-load aggregates, OX, F43121, F0911, F554201T, SBXUSSSAND, F0101, F4201) ──
    # `la` carries bol / sand_ticket / sbx_weight / load_last_status(+MX,MI) / the leg-B back-fill / the
    # ex-NOT-EXISTS flag — one join now replaces the old bol + sandtkt + sxw + lls joins.
    j = (base.alias("sd")
         .join(la, (F.col("sd.l_kcoo") == F.col("_la_kcoo")) &
                   (F.col("sd.l_doco") == F.col("_la_doco")) &
                   (F.col("sd.l_dcto") == F.col("_la_dcto")), "left")
         .join(ox, (F.col("sd.l_litm") == F.col("_ox_item")) &
                   (F.col("sd.l_doco") == F.col("_ox_doco")), "left")
         .join(recv, (F.col("sd.l_cars") == F.col("_rc_pran8")) &
                     (F.col("sd.l_kcoo") == F.col("_rc_kcoo")) &
                     (F.col("sd.l_doco") == F.col("_rc_doco")) &
                     (F.col("sd.l_litm") == F.col("_rc_item")), "left")
         .join(glpost, (F.col("sd.l_cars") == F.col("_gd_pran8")) &
                       (F.col("sd.l_kcoo") == F.col("_gd_kcoo")) &
                       (F.col("sd.l_doco") == F.col("_gd_doco")) &
                       (F.col("sd.l_litm") == F.col("_gd_item")), "left")
         .join(qcv, (F.col("sd.l_kcoo") == F.col("qc_kcoo")) &
                    (F.col("sd.l_doco") == F.col("qc_doco")) &
                    (F.col("sd.l_dcto") == F.col("qc_dcto")), "left")          # §4 join #1
         .join(sbxusssand, (F.col("sd.l_kcoo") == F.col("us_kcoo")) &
                           (F.col("sd.l_doco") == F.col("us_doco")) &
                           (F.col("sd.l_dcto") == F.col("us_dcto")), "left")   # §4 join #2
         .join(lofa, F.col("sd.l_vend") == F.col("ab_an8"), "left")            # §4 join #3 (LOFA=ABAN8)
         .join(hdr, (F.col("sd.l_kcoo") == F.col("h_kcoo")) &
                    (F.col("sd.l_doco") == F.col("h_doco")) &
                    (F.col("sd.l_dcto") == F.col("h_dcto")), "left"))   # report-5 header

    sel = j.select(
        # ── degenerate identifiers ──
        F.col("sd.l_doco").cast("long").alias("load_number"),       # SDDOCO (int64 — see design §7a)
        F.col("sd.l_dcto").alias("document_type"),                  # SDDCTO
        F.col("sd.l_kcoo").alias("company"),                        # SDKCOO
        F.col("sd.l_mcu").alias("district"),                        # SDMCU
        # ── address FK codes ──
        F.col("sd.l_an8").alias("sold_to"),                         # SDAN8
        F.col("sd.l_shan").alias("ship_to"),                        # SDSHAN
        F.col("sd.l_cars").alias("carrier"),                        # SDCARS
        F.col("sd.l_vend").alias("loading_facility"),               # LOFA=SDVEND
        # ── degenerate attributes ──
        F.col("sd.l_vr01").alias("customer_po"),                    # SDVR01
        F.col("sand_po_number"),                                    # F554201T QCDS50
        F.col("uss_customer_po"),                                   # SBXUSSSAND SOPONO
        F.col("sd.l_litm").alias("item_number"),                    # SDLITM
        F.col("sd.l_dsc1").alias("item_description"),               # SDDSC1
        F.col("sd.l_trdj").alias("order_date"),                     # SDTRDJ
        gl_date.alias("gl_date"),                                   # SDDGL (null→1900-01-01)
        F.col("uss_match"),
        F.col("uss_so_order_no"), F.col("uss_so_weight"), F.col("sbx_weight"), F.col("so_alt_bol_no"),
        F.col("sand_ticket"), F.col("bol"),
        F.col("sd.l_uom").alias("uom"),                             # SDUOM
        qty.alias("quantity"),                                      # QTY (derived)
        F.col("sd.l_uprc").alias("unit_price"),                     # SDUPRC (0 on PO_HOLADD)
        F.col("sd.l_aexp").alias("total_amount"),                   # SDAEXP (0 on PO_HOLADD)
        F.col("sd.l_lttr").alias("last_status"),                    # SDLTTR
        F.col("sd.l_nxtr").alias("next_status"),                    # SDNXTR
        F.col("sd.l_nxtr").cast("int").alias("next_status_num"),     # SDNXTR as int — core report SDNXTR<'581'
        F.col("sd.l_doc").cast("long").alias("invoice_number"),     # SDDOC (int64)
        # OX status — an orphan PO row reports ITS OWN PDLTTR/PDNXTR, not the load-level MAX
        F.coalesce(F.col("sd._pox_lttr"), F.col("_ox_lttr")).alias("ox_last_status"),
        F.coalesce(F.col("sd._pox_nxtr"), F.col("_ox_nxtr")).alias("ox_next_status"),
        ox_amount.alias("ox_amount"),                               # OXAMT (three-way rule above)
        ox_amount_gross.alias("ox_amount_gross"),                   # same money, NO item/status condition
        F.col("carrier_po_gl_post_flag"),                           # F0911
        F.col("po_receipt_gl_date"),                                # F43121
        # SDLNID — display the RAW JDE line number: 1.00 -> 1000.  Silver decoded the 3 implied decimals;
        # this puts them back, which is what Hubble shows. It is LOSSLESS *because* of the decimals, not
        # in spite of them: a fractional kit/component line 1.010 becomes 1010, so nothing is truncated —
        # the earlier `formatString: 0.###` workaround is no longer needed. round() first: 1.01 * 1000 is
        # 1009.9999999999999 in binary floating point, and a bare cast would floor it to 1009.
        F.round(F.col("sd.l_lnid") * F.lit(1000), 0).cast("long").alias("line_id"),
        # ── variation columns ──
        F.col("sd.row_class"),
        F.col("sd.l_lnty").alias("line_type"),                      # SDLNTY
        F.col("sd.l_prp1").alias("product_category"),               # SDPRP1
        F.col("sd.l_srp1").alias("sales_report_code_01"),           # SDSRP1
        F.col("sd.l_uorg").alias("units_ordered"),                  # SDUORG (raw decoded)
        F.col("sd.l_itwt").alias("item_weight"),                    # SDITWT
        F.col("sd.po_holadd_superseded"),                           # ex-NOT EXISTS (status → field)
        F.col("sd.l_po_dcto").alias("po_order_type"),                # PDDCTO (PO rows only)
        F.col("load_last_status"),                                  # report-5 per-load SDLTTR CASE
        F.col("load_max_last_status"), F.col("load_min_last_status"),   # MXLTTR / MILTTR (the CASE inputs)
        F.col("leg_1"), F.col("leg_2"), F.col("leg_3"), F.col("qc_string_3"),
        F.col("header_district"), F.col("header_sold_to"), F.col("header_ship_to"),
        F.col("header_carrier"), F.col("header_customer_po"), F.col("header_order_date"),
        # ── measure ──
        (F.col("_aburat") * F.lit(RATE_FACTOR)).alias("lofa_rate"),  # ABURAT (Hubble *0.01)
    ).distinct()          # == Hubble inner SELECT DISTINCT

    # ── Hubble GROUP BY (outer): one row per display tuple; SUM the single measure ──
    agg = (sel.groupBy(*FACT_GROUP_BY_COLS)
           .agg(F.sum("lofa_rate").alias("lofa_rate")))

    df = (agg
          .withColumn("load_scope_key",
                      load_scope_expr("company", "document_type", "load_number"))   # CDC delete scope (the load)
          .withColumn("load_line_key", sk(*FACT_GROUP_BY_COLS)))
    df = df.dropDuplicates(["load_line_key"])
    return df.select("load_line_key", "load_scope_key", *FACT_BUSINESS_COLS)

StatementMeta(, d3ebc738-5c6a-420e-b066-84b1d4d5928b, 7, Finished, Available, Finished, False)

In [5]:
# In[5]:


# =============================================================================
# FULL LOAD vs RESUME  (streaming approach from ESO4)
#   1) Stop any of our streams left alive from a previous run in this session.
#   2) FULL LOAD when OVERWRITE, the fact is missing, or a checkpoint is incomplete: drop + rebuild
#      the fact, snapshot F4211's AND F4311's Delta versions as init_ver, clear checkpoints.
#   3) Otherwise RESUME (init_ver = -1; the committed checkpoint offsets drive).
#   Address dims are REUSED (existing rpt.dim_address_book role views); dim_uss_plant is built by
#   nb_eso5_gold_dim_uss_plant (PREREQUISITE — read here for the SOORDERNO match MCU).
#   F554201T / F0911 / F43121 / F0101 / F4201 are STATIC snapshots. F0005 is NOT read here.
# =============================================================================
STREAMED = [F4211_TBL, F4311_TBL]      # both contribute ROWS ⇒ both must drive a recompute
_CKPT_PATHS = [f"{CKPT}/fact__{t}" for t in STREAMED]

def _checkpoints_exist():
    """True iff EVERY per-stream checkpoint has a COMMITTED offset (offsets/ non-empty), so an
    incomplete checkpoint forces a FULL LOAD rather than cold-starting a CDF reader at
    startingVersion=0 (v0 predates CDF enablement -> DELTA_MISSING_CHANGE_DATA)."""
    for p in _CKPT_PATHS:
        try:
            if not mssparkutils.fs.ls(f"{p}/offsets"):
                return False
        except Exception:
            return False
    return True

_STREAM_NAMES = {"fact__" + t for t in STREAMED}
_stopped = []
for _q in list(spark.streams.active):
    if _q.name in _STREAM_NAMES:
        _q.stop()
        _stopped.append(_q.name)
if _stopped:
    print(f"Stopped leftover streams: {_stopped}")

_FULL_LOAD = OVERWRITE or not spark.catalog.tableExists(T_FACT) or not _checkpoints_exist()

if _FULL_LOAD:
    print("== FULL LOAD ==")
    spark.sql(f"DROP TABLE IF EXISTS {T_FACT}")
    _write_new_table(transform_fact(), T_FACT)
    print(f"  ✓ seeded {T_FACT}")
    _init_ver = {t: current_version(t) for t in STREAMED}
    print(f"  init versions: {_init_ver}")
    try:
        mssparkutils.fs.rm(CKPT, True)
        print("  checkpoints cleared")
    except Exception as e:
        print(f"  checkpoint clear skipped: {e}")
    print("✓ full load complete")
else:
    print("== RESUME from checkpoint ==")
    _init_ver = {}

StatementMeta(, deefa956-1ec2-4a24-9383-e4fa8e821717, 7, Finished, Available, Finished, False)

== FULL LOAD ==
  ✓ seeded lh_jde_gold.rpt.fact_extended_sales_order_5
  init versions: {'f4211_sales_order_detail_file': 2500, 'f4311_purchase_order_detail_file': 742}
  checkpoints cleared
✓ full load complete


In [8]:
# In[6]:


# =============================================================================
# STREAM BATCH HANDLERS  (structure from ESO4)
#   Map each source's change rows to the affected LOAD scope and recompute_fact() that scope.
#   F4211 change rows carry the load keys directly. F4311 (OX purchase orders) carries the SAME
#   document number as the sales load — PDDOCO = SDDOCO — but its own order_type is 'OX', so the scope
#   must be re-stamped to the SX load, which is what the fact is keyed by.
# =============================================================================
def _changed(batch_df, init_ver):
    if batch_df.rdd.isEmpty():
        return None
    if init_ver >= 0:
        batch_df = batch_df.filter(F.col("_commit_version") > init_ver)
    if batch_df.rdd.isEmpty():
        return None
    return batch_df.filter(F.col("_change_type").isin("insert", "update_postimage", "delete"))

def make_fact_f4211_handler(init_ver):
    def handler(batch_df, batch_id):
        ch = _changed(batch_df, init_ver)
        if ch is None:
            return
        loads = ch.select(F.col("company_key_order_no"), F.col("order_type"),
                          F.col("document_order_invoice_e")).distinct()
        n = recompute_fact(loads)   # delete load scope + append recomputed lines (self-locks)
        print(f"[f4211] fact batch={batch_id} rows={n}")
    return handler

def make_fact_f4311_handler(init_ver):
    def handler(batch_df, batch_id):
        ch = _changed(batch_df, init_ver)
        if ch is None:
            return
        # Any F4311 change is routed to its SX load — no order-type or company filter. Recomputing a load
        # that turns out not to be affected is merely wasted work; MISSING one silently staler the fact.
        loads = (ch.select(F.col("company_key_order_no"),
                           F.lit("SX").alias("order_type"),    # re-stamp the PO → the SX load it belongs to
                           F.col("document_order_invoice_e")).distinct())
        n = recompute_fact(loads)
        print(f"[f4311] fact batch={batch_id} rows={n}")
    return handler

_HANDLERS = {F4211_TBL: make_fact_f4211_handler, F4311_TBL: make_fact_f4311_handler}

StatementMeta(, deefa956-1ec2-4a24-9383-e4fa8e821717, 10, Finished, Available, Finished, True)

In [7]:
# In[7]:


# =============================================================================
# START STREAMS — Silver Change Data Feed -> foreachBatch -> CDC write, every 30 s.
# REQUIRES delta.enableChangeDataFeed = true on BOTH sources (F4211 and F4311).
# =============================================================================
def _start_ver(iv, tbl):
    """Full load: init_ver (exists, carries CDF; handler skips <= it). Resume (iv < 0): fall back
    to the source's CURRENT version, never 0 (v0 predates CDF enablement)."""
    return iv if iv >= 0 else current_version(tbl)

for _tbl in STREAMED:
    _iv = _init_ver.get(_tbl, -1)
    _sv = _start_ver(_iv, _tbl)
    (spark.readStream.format("delta")
         .option("readChangeFeed",  "true")
         .option("startingVersion", _sv)
         .table(sname(_tbl))
     .writeStream
         .foreachBatch(_HANDLERS[_tbl](_iv))
         .option("checkpointLocation", f"{CKPT}/fact__{_tbl}")
         .trigger(**TRIGGER)
         .queryName("fact__" + _tbl)
         .start())
    print(f"  fact__{_tbl}  startingVersion={_sv}  init_ver={_iv}")

print(f"== started {len(STREAMED)} streams — continuous, trigger {TRIGGER}. Target {T_FACT}. "
      "ONE fact serves the core report + all 4 Filter-Capture variations (filter on row_class). "
      "Address dims reused (rpt.dim_address_book role views). ==")
spark.streams.awaitAnyTermination()

StatementMeta(, deefa956-1ec2-4a24-9383-e4fa8e821717, 9, Finished, Available, Finished, False)

  fact__f4211_sales_order_detail_file  startingVersion=2500  init_ver=2500
  fact__f4311_purchase_order_detail_file  startingVersion=742  init_ver=742
== started 2 streams — continuous, trigger {'processingTime': '30 seconds'}. Target lh_jde_gold.rpt.fact_extended_sales_order_5. ONE fact serves the core report + all 4 Filter-Capture variations (filter on row_class). Address dims reused (rpt.dim_address_book role views). ==


StreamingQueryException: [STREAM_FAILED] Query [id = 3d28b5f5-492e-43b1-850c-af7571dd2d2b, runId = a1002d5f-2819-4ffb-b4ca-91b5b8c9e441] terminated with exception: [DELTA_MISSING_CHANGE_DATA] Error getting change data for range [742 , 743] as change data was not
recorded for version [742]. If you've enabled change data feed on this table,
use `DESCRIBE HISTORY` to see when it was first enabled.
Otherwise, to start recording change data, use `ALTER TABLE table_name SET TBLPROPERTIES
(delta.enableChangeDataFeed=true)`.